# Genomics of Drug Sensitivity in Cancer (GDSC) — Analysis Notebook

In [64]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning / dimensionality reduction
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures

from sklearn.feature_selection import SelectKBest

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from xgboost import XGBRegressor

# Clustering

# Styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [65]:
TARGET_COLUMN = 'LN_IC50'
RANDOM_SEED = 42

## 1. Data Loading

In [66]:
gdsc = pd.read_excel("../data/GDSC.xlsx")

In [67]:
gdsc.head(5)

,COSMIC_ID,CELL_LINE_NAME,TCGA_DESC,DRUG_ID,DRUG_NAME,LN_IC50,AUC,Z_SCORE,GDSC Tissue descriptor 1,GDSC Tissue descriptor 2,Cancer Type (matching TCGA label),Microsatellite instability Status (MSI),Screen Medium,Growth Properties,CNA,Gene Expression,Methylation,TARGET,TARGET_PATHWAY
0,683667,PFSK-1,MB,1003,Camptothecin,-1.463887,0.930220,0.433123,nervous_system,medulloblastoma,MB,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
1,687448,COLO-829,SKCM,1003,Camptothecin,-1.235034,0.867348,0.557727,skin,melanoma,SKCM,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
2,687455,RT4,BLCA,1003,Camptothecin,-2.963191,0.821438,-0.383200,urogenital_system,Bladder,BLCA,MSS/MSI-L,D/F12,Adherent,Y,Y,Y,TOP1,DNA replication
3,687457,SW780,BLCA,1003,Camptothecin,-1.449138,0.905050,0.441154,urogenital_system,Bladder,BLCA,MSS/MSI-L,D/F12,Adherent,Y,Y,Y,TOP1,DNA replication
4,687459,TCCSUP,BLCA,1003,Camptothecin,-2.350633,0.843430,-0.049682,urogenital_system,Bladder,BLCA,MSS/MSI-L,D/F12,Adherent,Y,Y,Y,TOP1,DNA replication


In [68]:
gdsc.dtypes

COSMIC_ID                                    int64
CELL_LINE_NAME                              object
TCGA_DESC                                   object
DRUG_ID                                      int64
DRUG_NAME                                   object
LN_IC50                                    float64
AUC                                        float64
Z_SCORE                                    float64
GDSC Tissue descriptor 1                    object
GDSC Tissue descriptor 2                    object
Cancer Type (matching TCGA label)           object
Microsatellite instability Status (MSI)     object
Screen Medium                               object
Growth Properties                           object
CNA                                         object
Gene Expression                             object
Methylation                                 object
TARGET                                      object
TARGET_PATHWAY                              object
dtype: object

## MACHINE LEARNING MODELING - PREDICTING LN_IC50

### Preprocessing

Identifiers convey not meaningful biological information, therefore they should be excluded from the dataset. **AUC**, and **Z_SCORE** are alternative measurements of the same information captured by **LN_IC50**. To prevent the introduction of data leakage, they will be exluded as well.

In [69]:
to_excludes = ["COSMIC_ID", "DRUG_ID", "Z_SCORE", "AUC", "Cancer Type (matching TCGA label)"]
df = gdsc.copy()
df.drop(columns=to_excludes, inplace=True)

### SPLIT THE DATASET: TRAINING AND TEST SETS

In [70]:
y = df[TARGET_COLUMN]
X = df.drop(columns=TARGET_COLUMN)

print("Features Set:")
display(X.head(2))

print("\nTarget Set:")
display(y.head(2))

Features Set:


,CELL_LINE_NAME,TCGA_DESC,DRUG_NAME,GDSC Tissue descriptor 1,GDSC Tissue descriptor 2,Microsatellite instability Status (MSI),Screen Medium,Growth Properties,CNA,Gene Expression,Methylation,TARGET,TARGET_PATHWAY
0,PFSK-1,MB,Camptothecin,nervous_system,medulloblastoma,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
1,COLO-829,SKCM,Camptothecin,skin,melanoma,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication



Target Set:


0   -1.463887
1   -1.235034
Name: LN_IC50, dtype: float64

In [71]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=RANDOM_SEED)

### ENCODING

In [72]:
categorical_columns = X_train.select_dtypes(exclude='number').columns

# Fit encoder on training data
ohe = OneHotEncoder(
    sparse_output=False,
    drop='first',
    handle_unknown='error'
)

encoded_train = ohe.fit_transform(X_train[categorical_columns])
encoded_test = ohe.transform(X_test[categorical_columns])

encoded_feature_names = ohe.get_feature_names_out(categorical_columns)


X_train_enc = pd.DataFrame(
    encoded_train,
    columns=encoded_feature_names
)

X_test_enc = pd.DataFrame(
    encoded_test,
    columns=encoded_feature_names
)

print("Encoded training set")
display(X_train_enc.head())

print("Encoded test set")
display(X_test_enc.head())

Encoded training set


,CELL_LINE_NAME_23132-87,CELL_LINE_NAME_42-MG-BA,CELL_LINE_NAME_451Lu,CELL_LINE_NAME_639-V,CELL_LINE_NAME_647-V,CELL_LINE_NAME_769-P,CELL_LINE_NAME_786-0,CELL_LINE_NAME_8-MG-BA,CELL_LINE_NAME_8305C,CELL_LINE_NAME_8505C,...,TARGET_PATHWAY_JNK and p38 signaling,TARGET_PATHWAY_Metabolism,TARGET_PATHWAY_Mitosis,TARGET_PATHWAY_Other,"TARGET_PATHWAY_Other, kinases",TARGET_PATHWAY_PI3K/MTOR signaling,TARGET_PATHWAY_Protein stability and degradation,TARGET_PATHWAY_RTK signaling,TARGET_PATHWAY_WNT signaling,TARGET_PATHWAY_p53 pathway
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


Encoded test set


,CELL_LINE_NAME_23132-87,CELL_LINE_NAME_42-MG-BA,CELL_LINE_NAME_451Lu,CELL_LINE_NAME_639-V,CELL_LINE_NAME_647-V,CELL_LINE_NAME_769-P,CELL_LINE_NAME_786-0,CELL_LINE_NAME_8-MG-BA,CELL_LINE_NAME_8305C,CELL_LINE_NAME_8505C,...,TARGET_PATHWAY_JNK and p38 signaling,TARGET_PATHWAY_Metabolism,TARGET_PATHWAY_Mitosis,TARGET_PATHWAY_Other,"TARGET_PATHWAY_Other, kinases",TARGET_PATHWAY_PI3K/MTOR signaling,TARGET_PATHWAY_Protein stability and degradation,TARGET_PATHWAY_RTK signaling,TARGET_PATHWAY_WNT signaling,TARGET_PATHWAY_p53 pathway
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [80]:
X_train_enc.shape

(129682, 1269)

## FEATURE SELECTION

In [ ]:
# # %%
# from sklearn.feature_selection import SelectKBest, f_regression

# # Number of top features to keep
# K = 500

# # Initialize selector
# selector = SelectKBest(
#     score_func=f_regression,
#     k=K
# )

# # Fit on training data only
# X_train_selected = selector.fit_transform(X_train_enc, y_train)

# # Apply same transformation to test data
# X_test_selected = selector.transform(X_test_enc)

# # Selected feature names
# selected_features = X_train_enc.columns[selector.get_support()]

# print(f"Number of selected features: {len(selected_features)}")

# print("\nTop Selected Features:")
# display(selected_features)

Number of selected features: 500

Top Selected Features:


Index(['CELL_LINE_NAME_A3-KAW', 'CELL_LINE_NAME_A4-Fuk',
       'CELL_LINE_NAME_ALL-PO', 'CELL_LINE_NAME_ALL-SIL',
       'CELL_LINE_NAME_AMO-1', 'CELL_LINE_NAME_ATN-1', 'CELL_LINE_NAME_AsPC-1',
       'CELL_LINE_NAME_BC-1', 'CELL_LINE_NAME_BE-13', 'CELL_LINE_NAME_BT-474',
       ...
       'TARGET_PATHWAY_JNK and p38 signaling', 'TARGET_PATHWAY_Metabolism',
       'TARGET_PATHWAY_Mitosis', 'TARGET_PATHWAY_Other',
       'TARGET_PATHWAY_Other, kinases', 'TARGET_PATHWAY_PI3K/MTOR signaling',
       'TARGET_PATHWAY_Protein stability and degradation',
       'TARGET_PATHWAY_RTK signaling', 'TARGET_PATHWAY_WNT signaling',
       'TARGET_PATHWAY_p53 pathway'],
      dtype='object', length=500)

## MODELING

In [74]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

xgb = XGBRegressor()

In [75]:
rf.fit(X_train_enc, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [76]:
rf.score(X_test_enc, y_test)

0.8006930844011229

In [77]:
predictions = rf.predict(X_test_enc)

In [78]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
X_test_enc
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

MAE : 0.9321
RMSE: 1.6117
R²  : 0.8007


In [ ]:
from sklearn.model_selection import GridSearchCV


params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}


estimator = RandomForestRegressor(random_state=RANDOM_SEED)


grid = GridSearchCV(estimator= estimator, param_grid=params, cv=5, scoring='neg_mean_absolute_error')


grid.fit(X_train_enc, y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best score: {grid.best_score_}")


best_model = grid.best_estimator_